
# Qwen 3‑4B Fine‑tuning on Kaggle (T4 ×2) with Unsloth + LoRA/QLoRA — Banking Advisor (CSV)

This notebook fine‑tunes a **Qwen 3/2.5 4B Instruct** model using **Unsloth** with **LoRA/QLoRA**.  
It expects your dataset as a CSV with at least these columns:
- `instruction` — the user question
- `response` — the assistant answer

You can add any other columns (e.g., `tags, category, intent, …`), they will be ignored by the trainer.

**Highlights**
- 4‑bit QLoRA for low VRAM on **T4 ×2** (Kaggle P100/T4 often handle this).
- Proper **chat templating** for Qwen.
- **Mask loss on assistant part only** (using TRL's `DataCollatorForCompletionOnlyLM`).
- Every step prints logs: GPU status, dataset stats, sample rows, and training progress.
- Saves LoRA adapters to `/kaggle/working` for download at the end.


In [2]:

# %% [markdown]
# ## 1) Runtime & GPU check

import os, sys, platform, subprocess, json, time, textwrap
from datetime import datetime

def log_header(title):
    print(f"\n=== {title} @ {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} ===")

def gpu_stats():
    try:
        import torch
        ngpu = torch.cuda.device_count()
        print(f"PyTorch: {torch.__version__}")
        print(f"CUDA available: {torch.cuda.is_available()} | GPUs: {ngpu}")
        for i in range(ngpu):
            name = torch.cuda.get_device_name(i)
            mem_total = torch.cuda.get_device_properties(i).total_memory / (1024**3)
            mem_free, mem_total2 = torch.cuda.mem_get_info(i)
            print(f" - GPU{i}: {name} | VRAM: {mem_total:.2f} GB | Free (approx): {mem_free/(1024**3):.2f} GB")
    except Exception as e:
        print("GPU stats error:", e)

log_header("Environment")
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working dir:", os.getcwd())

log_header("nvidia-smi")
try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as e:
    print("nvidia-smi error:", e)

log_header("GPU Stats")
gpu_stats()
print("✅ Runtime & GPU check complete.")



=== Environment @ 2025-09-20 06:33:43 ===
Python: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
Platform: Linux-6.6.56+-x86_64-with-glibc2.35
Working dir: /kaggle/working

=== nvidia-smi @ 2025-09-20 06:33:43 ===
Sat Sep 20 06:33:43 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C  

In [2]:
!pip uninstall -y triton
!pip install "triton==3.0.0"

Found existing installation: triton 2.1.0
Uninstalling triton-2.1.0:
  Successfully uninstalled triton-2.1.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.4/209.4 MB 102.7 MB/s  0:00:020:00:0100:01


In [1]:
!pip install unsloth
!pip install transformers==4.55.4
!pip install --no-deps trl==0.22.2


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 2.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.6/314.6 kB 8.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 563.4/563.4 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 93.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.9/233.9 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 15.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:

# %% [markdown]
# ## 3) Configuration (edit these as needed)

from dataclasses import dataclass

log_header("Config")

@dataclass
class CFG:
    # Path to your CSV on Kaggle (adjust after uploading your dataset)
    CSV_PATH: str = "/kaggle/input/data-banking-processed/final_sua_mapped_v2.csv"  # <-- EDIT THIS

    # Base model (pick one). Qwen/Qwen2.5-4B-Instruct is official and works with Unsloth.
    # If you have an Unsloth-prequantized variant you prefer, set it here instead.
    MODEL_ID: str = "unsloth/Qwen3-4B-Instruct-2507"

    # Max sequence length (Qwen supports long contexts; 2048 is memory-safe for T4 x2)
    MAX_SEQ_LEN: int = 2048

    # LoRA
    LORA_R: int = 16            # rank (16 is safer on small GPUs; you may try 32 if headroom permits)
    LORA_ALPHA: int = 16
    LORA_DROPOUT: float = 0.05
    TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

    # Training
    PER_DEVICE_TRAIN_BATCH_SIZE: int = 2   # per-GPU batch
    GRAD_ACCUM: int = 8                    # effective batch = bs * grad_accum * num_gpus
    LR: float = 2e-4
    WEIGHT_DECAY: float = 0.01
    NUM_EPOCHS: int = 1                    # increase for real training (e.g., 2-3)
    WARMUP_RATIO: float = 0.03
    LOGGING_STEPS: int = 5
    SAVE_STEPS: int = 0                    # disable frequent saves; we'll save at end
    EVAL_RATIO: float = 0.0                # set >0.0 if you want a validation split

    # Compute dtype
    FP16: bool = True                      # T4 supports fp16; bf16 is not supported on T4

    # Output
    OUT_DIR: str = "/kaggle/working/qwen3_4b_banking_lora"

cfg = CFG()
print(cfg)
print("✅ Config loaded.")



=== Config @ 2025-09-20 06:33:46 ===
CFG(CSV_PATH='/kaggle/input/data-banking-processed/final_sua_mapped_v2.csv', MODEL_ID='unsloth/Qwen3-4B-Instruct-2507', MAX_SEQ_LEN=2048, LORA_R=16, LORA_ALPHA=16, LORA_DROPOUT=0.05, PER_DEVICE_TRAIN_BATCH_SIZE=2, GRAD_ACCUM=8, LR=0.0002, WEIGHT_DECAY=0.01, NUM_EPOCHS=1, WARMUP_RATIO=0.03, LOGGING_STEPS=5, SAVE_STEPS=0, EVAL_RATIO=0.0, FP16=True, OUT_DIR='/kaggle/working/qwen3_4b_banking_lora')
✅ Config loaded.


In [4]:

# %% [markdown]
# ## 4) Imports & helpers

log_header("Imports")
import os, gc, math, random
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
import torch
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from trl import SFTTrainer, SFTConfig
# 1) cố gắng dùng TRL nếu có
try:
    from trl import DataCollatorForCompletionOnlyLM  # TRL mới
except Exception:
    try:
        from trl.trainer.utils import DataCollatorForCompletionOnlyLM  # TRL 0.9.x
    except Exception:
        DataCollatorForCompletionOnlyLM = None

# 2) nếu vẫn không có, dùng collator fallback tương đương:
if DataCollatorForCompletionOnlyLM is None:
    from dataclasses import dataclass
    from typing import List, Dict, Any
    from transformers import PreTrainedTokenizerBase

    def _find_subsequence(seq: List[int], sub: List[int]) -> int:
        """Tìm vị trí bắt đầu của 'sub' trong 'seq'; không thấy → -1."""
        if not sub or len(sub) > len(seq): 
            return -1
        # đơn giản & đủ nhanh cho batch nhỏ
        for i in range(len(seq) - len(sub) + 1):
            if seq[i:i+len(sub)] == sub:
                return i
        return -1

    @dataclass
    class CompletionOnlyCollator:
        tokenizer: PreTrainedTokenizerBase
        response_template: str

        def __post_init__(self):
            # tokenize template 1 lần (không thêm special tokens)
            self.template_ids = self.tokenizer.encode(
                self.response_template, add_special_tokens=False
            )

        def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
            # Kỳ vọng mỗi feature đã có "input_ids" (đã tokenized)
            # Nếu bạn đang truyền text, bạn có thể map trước bằng tokenizer.
            batch = self.tokenizer.pad(
                features,
                padding=True,
                return_tensors="pt",
            )
            input_ids = batch["input_ids"]
            labels = input_ids.clone()

            # Mặc định mask hết
            labels[:] = -100

            # Tìm vị trí template và bật loss kể từ đó
            for i in range(input_ids.size(0)):
                ids = input_ids[i].tolist()
                pos = _find_subsequence(ids, self.template_ids)
                if pos == -1:
                    # nếu không tìm thấy template: cho học toàn bộ (hoặc giữ -100 tuỳ bạn)
                    # Ở đây: học toàn bộ như phương án "lenient"
                    labels[i] = input_ids[i]
                else:
                    labels[i, pos:] = input_ids[i, pos:]

            batch["labels"] = labels
            return batch

    def DataCollatorForCompletionOnlyLM(response_template: str, tokenizer: PreTrainedTokenizerBase):
        # Trả về instance để API giống TRL
        return CompletionOnlyCollator(tokenizer=tokenizer, response_template=response_template)


def print_gpu():
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            free, total = torch.cuda.mem_get_info(i)
            print(f"GPU{i} free: {free/(1024**3):.2f} GB / total approx: {total/(1024**3):.2f} GB")

def sample_df(df, k=3):
    try:
        display(df.sample(min(k, len(df)), random_state=42))
    except:
        print(df.head(min(k, len(df))))

print_gpu()
print("✅ Imports ready.")



=== Imports @ 2025-09-20 06:33:46 ===
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-09-20 06:33:53.715782: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758350033.921071      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758350033.981742      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!
GPU0 free: 14.64 GB / total approx: 14.74 GB
GPU1 free: 14.64 GB / total approx: 14.74 GB
✅ Imports ready.


In [9]:
import importlib, pkgutil, torch
print("Torch:", torch.__version__, "| torch.version.cuda:", torch.version.cuda)

spec = importlib.util.find_spec("triton")
print("Has triton:", spec is not None)
if spec:
    import triton
    print("triton:", getattr(triton, "__version__", "unknown"))
    print("has triton.ops:", pkgutil.find_loader("triton.ops") is not None)


Torch: 2.8.0+cu128 | torch.version.cuda: 12.8
Has triton: True
triton: 3.4.0
has triton.ops: False


In [5]:

# %% [markdown]
# ## 5) Load tokenizer & model (QLoRA 4‑bit via Unsloth)

log_header("Load model/tokenizer")
device_count = torch.cuda.device_count()
print(f"Detected {device_count} GPU(s).")

# Lấy trực tiếp model + tokenizer từ Unsloth (trả về tuple)
model, tokenizer = FastLanguageModel.from_pretrained(
    cfg.MODEL_ID,
    max_seq_length=cfg.MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,          # để Unsloth tự chọn kernel
    device_map="auto",
)
# Đảm bảo EOS
if tokenizer.eos_token is None:
    tokenizer.eos_token = tokenizer.pad_token or "</s>"

print("Base model loaded (4-bit).")

# Bật tối ưu training của Unsloth
FastLanguageModel.for_training(model, use_gradient_checkpointing="unsloth")
print("Enabled Unsloth gradient checkpointing.")

# Gắn LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=cfg.LORA_R,
    lora_alpha=cfg.LORA_ALPHA,
    lora_dropout=cfg.LORA_DROPOUT,
    target_modules=cfg.TARGET_MODULES,
    bias="none",
    use_rslora=True,
)
print("LoRA adapters attached.")
print_gpu()
print("✅ Model ready.")




=== Load model/tokenizer @ 2025-09-20 06:34:19 ===
Detected 2 GPU(s).
==((====))==  Unsloth 2025.9.7: Fast Qwen3 patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Base model loaded (4-bit).
Enabled Unsloth gradient checkpointing.


Unsloth 2025.9.7 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


LoRA adapters attached.
GPU0 free: 13.20 GB / total approx: 14.74 GB
GPU1 free: 11.54 GB / total approx: 14.74 GB
✅ Model ready.


In [6]:

# %% [markdown]
# ## 6) Load & validate CSV

log_header("Load CSV")
import os
assert os.path.exists(cfg.CSV_PATH), f"CSV not found at {cfg.CSV_PATH}. Please upload and update CFG.CSV_PATH."

raw = pd.read_csv(cfg.CSV_PATH)
print("Columns:", list(raw.columns))
need = {"instruction","response"}
missing = need - set(c.lower() for c in raw.columns)
if missing:
    # Try case-insensitive match
    cols_lower = {c.lower(): c for c in raw.columns}
    if "instruction" in cols_lower and "response" in cols_lower:
        raw = raw.rename(columns={cols_lower["instruction"]:"instruction", cols_lower["response"]:"response"})
    else:
        raise ValueError(f"CSV must contain columns {need}. Missing (case-insensitive): {missing}")

# Basic cleaning
df = raw.copy()
df["instruction"] = df["instruction"].astype(str).str.strip()
df["response"]    = df["response"].astype(str).str.strip()
df = df.dropna(subset=["instruction","response"])
df = df[(df["instruction"]!="") & (df["response"]!="")]

print(f"Loaded rows: {len(df)}")
print("Sample:")
sample_df(df, 3)
print("✅ CSV loaded & validated.")



=== Load CSV @ 2025-09-20 06:34:51 ===
Columns: ['tags', 'instruction', 'category', 'intent', 'response']
Loaded rows: 25545
Sample:


,tags,instruction,category,intent,response
6628,BCPZ,"Tôi muốn hủy khoản vay, giúp tôi",LOAN,cancel_loan,Tôi xin lỗi khi nghe rằng bạn đang tìm cách hủ...
20099,BCLPZ,"Tôi muốn tìm một khu vực gần tôi, tôi cần sự g...",FIND,find_branch,Tôi ở đây để giúp bạn tìm một chi nhánh gần vị...
11548,BCIZ,"Tôi cần thông tin về hóa đơn của tôi, làm thế ...",FEES,check_fees,Chắc chắn! tôi ở đây để giúp bạn tìm thông tin...


✅ CSV loaded & validated.


In [7]:

# %% [markdown]
# ## 7) Train/Val split

log_header("Split")
if cfg.EVAL_RATIO and 0 < cfg.EVAL_RATIO < 0.5:
    eval_size = int(len(df)*cfg.EVAL_RATIO)
else:
    eval_size = 0

if eval_size > 0:
    df_train = df.iloc[:-eval_size].reset_index(drop=True)
    df_eval  = df.iloc[-eval_size:].reset_index(drop=True)
else:
    df_train, df_eval = df, None

print("Train size:", len(df_train))
print("Eval size :", 0 if df_eval is None else len(df_eval))
print("✅ Split done.")



=== Split @ 2025-09-20 06:34:52 ===
Train size: 25545
Eval size : 0
✅ Split done.


In [8]:

# %% [markdown]
# ## 8) Formatting to Qwen chat + response-only loss

log_header("Chat template & collator")

# Try to detect assistant delimiter used by tokenizer template
chat_tmpl = getattr(tokenizer, "chat_template", "")
print("Has tokenizer.chat_template:", bool(chat_tmpl))

ASSISTANT_OPEN = None
# Common Qwen templates use "<|im_start|>assistant" then content, ending with "<|im_end|>\n"
if "assistant" in chat_tmpl and "<|im_start|>" in chat_tmpl:
    ASSISTANT_OPEN = "<|im_start|>assistant"
elif "<|assistant|>" in chat_tmpl:
    ASSISTANT_OPEN = "<|assistant|>"
else:
    # Fallback to a safe guess for Qwen2.5 chat template
    ASSISTANT_OPEN = "<|im_start|>assistant"

print("Response template token guess:", ASSISTANT_OPEN)

def to_messages(row):
    # You can add system message if desired; for now we use user + assistant
    return [
        {"role": "user", "content": row["instruction"]},
        {"role": "assistant", "content": row["response"]},
    ]

def to_text(row):
    messages = to_messages(row)
    txt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,   # full conversation including assistant
    )
    return txt

# Create dataset with a single "text" column
train_ds = Dataset.from_pandas(df_train[["instruction","response"]].copy())
train_ds = train_ds.map(lambda ex: {"text": to_text(ex)}, remove_columns=train_ds.column_names)

if df_eval is not None:
    eval_ds = Dataset.from_pandas(df_eval[["instruction","response"]].copy())
    eval_ds = eval_ds.map(lambda ex: {"text": to_text(ex)}, remove_columns=eval_ds.column_names)
else:
    eval_ds = None

print("Example formatted text:\n", train_ds[0]["text"][:600])

# Collator that masks everything before the assistant token
collator = DataCollatorForCompletionOnlyLM(
    response_template=ASSISTANT_OPEN,
    tokenizer=tokenizer,
)
print("✅ Formatting & collator ready.")



=== Chat template & collator @ 2025-09-20 06:34:52 ===
Has tokenizer.chat_template: True
Response template token guess: <|im_start|>assistant


Map:   0%|          | 0/25545 [00:00<?, ? examples/s]

Example formatted text:
 <|im_start|>user
Tôi muốn thu hút một thẻ, bạn có thể giúp tôi không?<|im_end|>
<|im_start|>assistant
<think>

</think>

Tôi ở đây để giúp bạn với điều đó! kích hoạt thẻ của bạn là một bước quan trọng để bắt đầu và tận hưởng lợi ích của nó. Đây là cách bạn có thể kích hoạt thẻ của bạn:
1. Tìm hướng dẫn kích hoạt: Tùy thuộc vào người phát hành thẻ, bạn có thể tìm thấy hướng dẫn kích hoạt trên một thẻ gắn với thẻ đó, trong gói chào đón, hoặc trên trang web của người phát hành.
2. Tham quan trang web kích hoạt của người phát hành thẻ: Sử dụng máy tính hoặc thiết bị di động của bạn, mở một trình d
✅ Formatting & collator ready.


In [9]:

# %% [markdown]
# ## 9) Trainer setup

log_header("Trainer setup")
training_args = SFTConfig(
    output_dir=cfg.OUT_DIR,
    per_device_train_batch_size=cfg.PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=cfg.GRAD_ACCUM,
    #num_train_epochs=cfg.NUM_EPOCHS,
    max_steps = 200,
    learning_rate=cfg.LR,
    weight_decay=cfg.WEIGHT_DECAY,
    logging_steps=cfg.LOGGING_STEPS,
    save_steps=cfg.SAVE_STEPS,
    lr_scheduler_type="cosine",
    warmup_ratio=cfg.WARMUP_RATIO,
    fp16=cfg.FP16,
    bf16=False,
    max_seq_length=cfg.MAX_SEQ_LEN,
    packing=False,                   # for chat, keep per-sample structure
    report_to=[],                    # disable W&B etc. inside Kaggle
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collator,
    args=training_args,
    dataset_text_field="text",
)

print(trainer)
print_gpu()
print("✅ Trainer ready.")



=== Trainer setup @ 2025-09-20 06:34:58 ===


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/25545 [00:00<?, ? examples/s]

GPU0 free: 13.20 GB / total approx: 14.74 GB
GPU1 free: 11.54 GB / total approx: 14.74 GB
✅ Trainer ready.


In [10]:

# %% [markdown]
# ## 10) Train

log_header("Training start")
print(f"Total train samples: {len(train_ds)}")
if eval_ds is not None:
    print(f"Total eval samples : {len(eval_ds)}")

train_result = trainer.train()
print(train_result)
print_gpu()
print("✅ Training complete.")



=== Training start @ 2025-09-20 06:35:10 ===
Total train samples: 25545


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 25,545 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,4.716300
10,3.056700
15,2.567200
20,2.356300
25,2.190200
30,2.348600
35,2.754300
40,2.245600
45,1.718700
50,2.477300


TrainOutput(global_step=200, training_loss=2.111062321662903, metrics={'train_runtime': 2082.3454, 'train_samples_per_second': 1.537, 'train_steps_per_second': 0.096, 'total_flos': 1.898553088425984e+16, 'train_loss': 2.111062321662903, 'epoch': 0.1252642292335395})
GPU0 free: 9.20 GB / total approx: 14.74 GB
GPU1 free: 10.59 GB / total approx: 14.74 GB
✅ Training complete.


In [11]:

# %% [markdown]
# ## 11) Save LoRA adapters

log_header("Save adapters")
os.makedirs(cfg.OUT_DIR, exist_ok=True)
trainer.model.save_pretrained(cfg.OUT_DIR)
tokenizer.save_pretrained(cfg.OUT_DIR)
print("Saved to:", cfg.OUT_DIR)
print("Dir list:", os.listdir(cfg.OUT_DIR))
print("✅ Saved LoRA adapters.")



=== Save adapters @ 2025-09-20 07:13:16 ===
Saved to: /kaggle/working/qwen3_4b_banking_lora
Dir list: ['adapter_config.json', 'special_tokens_map.json', 'merges.txt', 'chat_template.jinja', 'adapter_model.safetensors', 'README.md', 'added_tokens.json', 'tokenizer.json', 'tokenizer_config.json', 'checkpoint-200', 'vocab.json']
✅ Saved LoRA adapters.


In [15]:

# %% [markdown]
# ## 12) [Optional] Merge LoRA into full model (fp16)
# WARNING: This consumes more VRAM and time. Enable only if you need a merged model for deployment.
# Set `DO_MERGE=True` to run.

log_header("Optional merge")
DO_MERGE = True

if DO_MERGE:
    from peft import AutoPeftModelForCausalLM
    base = AutoPeftModelForCausalLM.from_pretrained(
        cfg.OUT_DIR,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    merged = base.merge_and_unload()
    merge_dir = cfg.OUT_DIR + "_merged_fp16"
    os.makedirs(merge_dir, exist_ok=True)
    merged.save_pretrained(merge_dir, safe_serialization=True)
    tokenizer.save_pretrained(merge_dir)
    print("Merged model saved to:", merge_dir)
else:
    print("Skip merge. Set DO_MERGE=True to enable.")
print("✅ Merge step done (skipped by default).")



=== Optional merge @ 2025-09-20 07:15:29 ===


/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/bnb.py:351: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Merged model saved to: /kaggle/working/qwen3_4b_banking_lora_merged_fp16
✅ Merge step done (skipped by default).


In [24]:

# %% [markdown]
# ## 13) Quick inference sanity check

log_header("Sanity inference")

from transformers import TextStreamer

prompt = "Bạn có thể phân tích ngân hàng BIDV là gì không ?"
messages = [
    {"role": "user", "content": prompt}
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
with torch.no_grad():
    out = model.generate(
        input_ids=input_ids,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        streamer=streamer,
        eos_token_id=tokenizer.eos_token_id,
    )

print("\n\n✅ Inference check complete.")



=== Sanity inference @ 2025-09-20 07:52:20 ===
</tool_call>

</tool_call>

Tôi sẽ làm tốt nhất của tôi! tôi ở đây để giúp bạn phân tích ngân hàng BIDV. để phân tích một ngân hàng, chúng tôi cần xem xét một số yếu tố khác nhau. đây là một cách tiếp cận để phân tích ngân hàng BIDV:
1. Kinh doanh và tình hình tài chính: đánh giá tình hình tài chính của ngân hàng, bao gồm các khoản vay, lợi nhuận, và nợ. Điều này sẽ cho bạn một cái nhìn vào sức mạnh tài chính của họ và khả năng họ có thể đáp ứng các khoản vay.
2. Chiến lược cạnh tranh: xem xét các chiến lược cạnh tranh của ngân hàng BIDV, chẳng hạn như các sản phẩm tài chính, dịch vụ khách hàng, và giá cả. Điều này sẽ cho bạn biết họ có thể có lợi thế hoặc điểm yếu so với các đối thủ khác trong ngành.
3. Thị trường: phân tích thị trường của ngân hàng BIDV, chẳng hạn như số lượng khách hàng, thị phần, và xu hướng thị trường. Điều này sẽ cung cấp cho bạn một cái nhìn về sự phổ biến và sự phát triển của họ.
4. Quản lý: đánh giá quản lý ngân 

In [17]:
!zip -r file.zip /kaggle/working/qwen3_4b_banking_lora_merged_fp16

  adding: kaggle/working/qwen3_4b_banking_lora_merged_fp16/ (stored 0%)
  adding: kaggle/working/qwen3_4b_banking_lora_merged_fp16/model.safetensors (deflated 9%)
  adding: kaggle/working/qwen3_4b_banking_lora_merged_fp16/special_tokens_map.json (deflated 69%)
  adding: kaggle/working/qwen3_4b_banking_lora_merged_fp16/config.json (deflated 73%)
  adding: kaggle/working/qwen3_4b_banking_lora_merged_fp16/merges.txt (deflated 57%)
  adding: kaggle/working/qwen3_4b_banking_lora_merged_fp16/chat_template.jinja (deflated 76%)
  adding: kaggle/working/qwen3_4b_banking_lora_merged_fp16/added_tokens.json (deflated 68%)
  adding: kaggle/working/qwen3_4b_banking_lora_merged_fp16/tokenizer.json (deflated 81%)
  adding: kaggle/working/qwen3_4b_banking_lora_merged_fp16/tokenizer_config.json (deflated 90%)
  adding: kaggle/working/qwen3_4b_banking_lora_merged_fp16/vocab.json (deflated 61%)
  adding: kaggle/working/qwen3_4b_banking_lora_merged_fp16/generation_config.json (deflated 38%)



## 14) Next steps
- Increase `NUM_EPOCHS` (e.g., 2–3) and consider `EVAL_RATIO=0.05` for validation.
- If you hit VRAM limits, try smaller `LORA_R=8`, or reduce `PER_DEVICE_TRAIN_BATCH_SIZE=1` and raise `GRAD_ACCUM`.
- For strictly Vietnamese outputs, consider a small system prompt enforcing style/tone.
- If your CSV contains placeholders like `{{SUPPORT_PHONE}}`, keep them as-is; the model will learn to reproduce them.
